# Week 3 — RAG Pipeline (Google Colab driver)

End-to-end driver for the Week 3 RAG assignment. Runs all phases sequentially on a free Colab T4:

1. **Setup** — verify GPU, mount Drive, clone repo, symlink `data/` and `results/` to Drive, install deps
2. **Build indexes** — corpus, BM25, DPR embeddings (per dataset)
3. **Generate with Phi-3-mini (fp16)** — both datasets, all retrieval × K combinations
4. **Generate with Qwen2.5-7B-Instruct (4-bit)** — both datasets, all retrieval × K combinations
5. **Merge results** — build the EM/F1 table and failure cases

Outputs (`data/processed/*` and `results/*.csv`) are written to Google Drive so they survive runtime disconnects. All cells are idempotent: re-running the run-experiment cells skips already-completed (model × dataset × retrieval × K) combinations.

**Before running:** make sure your latest local changes (including the swap to `qwen`, the `quantize_4bit` flag, and `bitsandbytes` in `requirements.txt`) are committed and pushed to GitHub — the clone in cell 3 pulls from `origin/main`.

## 1. Setup

In [ ]:
# 1a. Confirm we have a GPU (should print a T4 row).
!nvidia-smi

In [ ]:
# 1b. Mount Google Drive — outputs land here so they survive disconnects.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1c. Clone the repo (or pull if it already exists), then chdir into the week 3 folder.
import os
from pathlib import Path

REPO_URL = "https://github.com/peter-altmayer/NLP_Applications_in_Research_and_Industry_SoSe_26.git"
REPO_DIR = Path("/content/repo")
WEEK3 = REPO_DIR / "nlp_week3-rag"

if REPO_DIR.exists():
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(WEEK3)
print("cwd:", os.getcwd())
assert (WEEK3 / "experiments" / "build_index.py").exists(), "build_index.py not found — did you push the latest commits?"

In [ ]:
# 1d. Symlink `data/` and `results/` inside the cloned repo to Drive so artifacts persist.
import shutil
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/nlp_week3-rag")
(DRIVE_ROOT / "data" / "processed").mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "data" / "raw").mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "results").mkdir(parents=True, exist_ok=True)

for sub in ("data", "results"):
    local = Path(sub)
    target = DRIVE_ROOT / sub
    if local.is_symlink():
        local.unlink()
    elif local.exists():
        shutil.rmtree(local)
    local.symlink_to(target, target_is_directory=True)
    print(f"  {sub}/  ->  {target}")

In [ ]:
# 1e. Install Python dependencies.
# Colab already ships torch + transformers + accelerate; we still pip install to pull rank_bm25 and bitsandbytes,
# and to upgrade transformers/accelerate to a recent version compatible with Qwen2.5 + 4-bit loading.
!pip install -q -r requirements.txt

### (Optional) Hugging Face login

Both generators chosen here (`microsoft/Phi-3-mini-4k-instruct` and `Qwen/Qwen2.5-7B-Instruct`) are **ungated** and need no token. The DPR models are also ungated. You only need to run the cell below if you later swap in a gated model (e.g. Mistral, Llama).

In [ ]:
# Uncomment to log in. Paste a read token from https://huggingface.co/settings/tokens.
# from huggingface_hub import login
# login()

## 2. Build indexes

Loads the first 1000 rows of each dataset, extracts the corpus (answer strings), builds the BM25 index, and encodes the corpus with both DPR context encoders. Outputs are saved to `data/processed/` (→ Drive).

Idempotent: DPR `.npy` files already on Drive are skipped (the BM25 pickle and corpus JSON are tiny and always rewritten).

In [ ]:
!python experiments/build_index.py

## 3. Generate with Phi-3-mini (fp16)

Phi-3-mini (3.8B) loads in fp16 on T4 without quantization. `batch_size=4` is comfortable. Each `(model, dataset)` run produces 10 CSVs in `results/` (K=0 once + 3 retrieval methods × 3 non-zero K).

In [ ]:
!python experiments/run_experiment.py --model phi3 --dataset trivia_qa --batch_size 4

In [ ]:
!python experiments/run_experiment.py --model phi3 --dataset natural_questions --batch_size 4

## 4. Generate with Qwen2.5-7B-Instruct (4-bit)

Qwen2.5-7B in fp16 (~14 GB) is too large for the T4's 16 GB once DPR encoders and the KV cache are loaded, so we load it in 4-bit NF4 via `bitsandbytes` (configured automatically because `qwen` is in `QUANTIZE_4BIT` in `run_experiment.py`).

`batch_size=2` is the safe default. If you see CUDA OOM, drop to 1.

In [ ]:
!python experiments/run_experiment.py --model qwen --dataset trivia_qa --batch_size 2

In [ ]:
!python experiments/run_experiment.py --model qwen --dataset natural_questions --batch_size 2

## 5. Merge and evaluate

Reads every per-run CSV, computes EM and token-F1 per row, writes `results/all_results.csv`, and prints the pivoted results table plus the 10 worst failure cases.

In [ ]:
!python experiments/merge_results.py | tee results/summary.txt

## Done

All artifacts are on Drive under `MyDrive/nlp_week3-rag/`. Pull them down to your local repo (`nlp_week3-rag/results/`) and commit `all_results.csv` + `summary.txt` for the submission.